In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
from tqdm.notebook import tqdm

In [ ]:
def verify_npy_file(file_path):
    """
    Checks a single .npy file for corruption, shape issues, and NaNs.
    Returns a status dict.
    """
    try:
        # Open in read-only memory map mode (doesn't load data yet)
        # This catches truncated or header-corrupted files immediately
        data = np.load(file_path, mmap_mode='r')
        
        # Check Shape (Should be 1D array of complex numbers)
        if len(data.shape) != 1:
            return {'status': 'BAD_SHAPE', 'details': f"Shape is {data.shape}"}
        
        # Check Data Type
        if not np.issubdtype(data.dtype, np.complexfloating):
            return {'status': 'BAD_DTYPE', 'details': f"Type is {data.dtype}"}
        
        # Check for NaNs/Infs 
        if np.isnan(data).any():
             return {'status': 'HAS_NAN', 'details': "Contains NaN values"}
             
        if np.isinf(data).any():
             return {'status': 'HAS_INF', 'details': "Contains Infinite values"}

        return {'status': 'OK', 'details': f"Samples: {data.shape[0]}"}

    except Exception as e:
        return {'status': 'CORRUPT', 'details': str(e)}

In [ ]:
npy_files = sorted(list(JAMSHIELD_NPY_DIR.glob("*.npy")))
verification_results = []

print(f"Auditing {len(npy_files)} .npy files...")
print("="*60)

# Run Checks
for f in tqdm(npy_files, desc="Checking NPYs"):
    res = verify_npy_file(f)
    
    # Store errors
    if res['status'] != 'OK':
        print(f"FAILED: {f.name} -> {res['status']} ({res['details']})")
        verification_results.append({
            'filename': f.name,
            'status': res['status'],
            'path': str(f)
        })

# Final Report
print("="*60)
if not verification_results:
    print(f"SUCCESS: All {len(npy_files)} files are valid, clean, and numeric-safe.")
else:
    print(f"WARNING: Found {len(verification_results)} bad files.")